In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
import scipy.stats as stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
from scipy.stats import skew 
from scipy.stats import kurtosis
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import FuncFormatter


In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'
#fpath = 'C:\\Users\\gianl\\Desktop\\Actigraphy Sara'

In [ ]:
df = pd.read_excel(fpath + '\\10.0_database_variables.xlsx', sheet_name = 'database_2022_2024')

In [ ]:
# Rename columns
df = df.rename(columns={'location(ita=0,uk=1)': 'location', 'week(1=free days)': 'weekday_type'})

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=(29, 5))
sns.boxplot(data=df['midsleep_h'], ax=ax[0])
sns.boxplot(data=df['sleep_start_decimal'], ax=ax[1])
sns.boxplot(data=df['sleep_end_decimal'], ax=ax[2])
sns.boxplot(data=df['phase(sleepoffset-sunrise)'], ax=ax[3])
sns.boxplot(data=df['sleep_duration(h)'], ax=ax[4])
sns.boxplot(data=df['waso(min)'], ax=ax[5])

plt.show()

In [ ]:
# define the start date
start_date = pd.to_datetime('2022-02-01')

In [ ]:
# function to count the week of the year from the start date 2022-02-01
def calculate_week_of_year(start_datetime): return (start_datetime - start_date).days // 7 + 5

# apply the function to calculate the week of the year
df['week_of_year'] = df['date'].apply(calculate_week_of_year)

In [ ]:
# adjust 'week of the year' to start from 0
df['week_of_year'] = df['week_of_year'] - 37

In [ ]:
# rename the location column as 0=ITA, 1=UK
df['location'] = df['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df['weekday_type'] = df['weekday_type'].map({0: 'work days', 1: 'free days'})

In [ ]:
# calculate the sleep duration for work days and free days
df['sleep_duration(h)'] = df['sleep_duration(h)'].astype(float)
df['sleep_duration_work_days'] = df['sleep_duration(h)'] * (df['weekday_type'] == 'work days')
df['sleep_duration_free_days'] = df['sleep_duration(h)'] * (df['weekday_type'] == 'free days')

In [ ]:
# filtered the midpoints by type of day of the week
# new dataframe with only the midpoints of the work days/free days
df_workdays = df[df['weekday_type'] == 'work days']
df_freedays = df[df['weekday_type'] == 'free days']

In [ ]:
# create a new df for weekly jetlag analysis
data_jetlag = df 

In [ ]:
# calculate the mean midpoint for each location, week and weekday
weekly_means_jetlag = data_jetlag.groupby(['location', 'week_of_year', 'weekday_type'])['midsleep_h'].mean().unstack()

In [ ]:
# calculate the jet lag 
weekly_means_jetlag['jet lag'] = weekly_means_jetlag['free days'] - weekly_means_jetlag['work days']

In [ ]:
# add a column with the location to the weekly_means_jetlag_UTC
weekly_means_jetlag['location'] = weekly_means_jetlag.index.get_level_values(0)

In [ ]:
# rename columns
df = df.rename(columns={'sleep_duration(h)': 'sleep_duration'})
df = df.rename(columns={'phase(sleepoffset-sunrise)': 'phase'})
df = df.rename(columns={'waso(min)': 'waso'})
df = df.rename(columns={'DST(0=ST)': 'DST_1'})

In [ ]:
# dictionary with the season dates
seasons = {'Winter': [(12, 21), (3, 20)], 'Spring': [(3, 21), (6, 20)], 'Summer': [(6, 21), (9, 22)], 'Autumn': [(9, 23), (12, 20)]}

df['date'] = pd.to_datetime(df['date'])

In [ ]:
# Function to get the season from the date
def get_season(date):
    month, day = date.month, date.day
    for season, ((start_month, start_day), (end_month, end_day)) in seasons.items():
        if (month == start_month and day >= start_day) or (month == end_month and day <= end_day):
            return season
        elif start_month < month < end_month:
            return season
    return 'Winter'  # for dates before 21st December and after 20th December

In [ ]:
# Applying the function to create a season column
df_workdays.loc[:, 'season'] = df_workdays['date'].apply(get_season)
df_freedays.loc[:, 'season'] = df_freedays['date'].apply(get_season)
df.loc[:, 'season'] = df['date'].apply(get_season)

In [ ]:
# new variable 'photoperiod' based on the location
# if column 'location' = 1 take the value from 'photoperiod (h, UK)' 
# if column 'location' = 0 then photoperiod (h, ITA)'
df['photoperiod'] = np.where(df['location'] == 'UK', df['photoperiod (h, UK)'], df['photoperiod (h, ITA)'])

In [ ]:
# add a column with the photoperiod for the UK and ITA
df_workdays.loc[df_workdays['location'] == 'UK', 'photoperiod'] = df_workdays['photoperiod (h, UK)'] 
df_workdays.loc[df_workdays['location'] == 'ITA', 'photoperiod'] = df_workdays['photoperiod (h, ITA)']

In [ ]:
# add the photoperiod column to the df_freedays
df_freedays.loc[df_freedays['location'] == 'UK', 'photoperiod'] = df_freedays['photoperiod (h, UK)']
df_freedays.loc[df_freedays['location'] == 'ITA', 'photoperiod'] = df_freedays['photoperiod (h, ITA)']

In [ ]:
# descpriptive statistics
all_descriptive = df.describe()
all_descriptive = all_descriptive.transpose()

In [ ]:
# add the index as a column
all_descriptive['variable'] = all_descriptive.index 

In [ ]:
#reset the index
all_descriptive = all_descriptive.reset_index(drop=True)

In [ ]:
all_descriptive.to_excel(fpath + '\\all_descriptive.xlsx')

In [ ]:
#descriptive for workdays and freedays
workdays_descriptive = df_workdays.describe()
workdays_descriptive = workdays_descriptive.transpose()

# add the index as a column
workdays_descriptive['variable'] = workdays_descriptive.index

#reset the index
workdays_descriptive = workdays_descriptive.reset_index(drop=True)

In [ ]:
# save the descriptive statistics for freedays
freedays_descriptive = df_freedays.describe()
freedays_descriptive = freedays_descriptive.transpose()

# add the index as a column
freedays_descriptive['variable'] = freedays_descriptive.index

#reset the index
freedays_descriptive = freedays_descriptive.reset_index(drop=True)

In [ ]:
# descpriptive statistics for ITA
descriptive_ita = df[df['location'] == 'ITA'].describe()
descriptive_ita = descriptive_ita.transpose()
descriptive_ita['variable'] = descriptive_ita.index 

In [ ]:
# descpriptive statistics for UK
descriptive_uk = df[df['location'] == 'UK'].describe()
descriptive_uk = descriptive_uk.transpose()
descriptive_uk['variable'] = descriptive_uk.index 

In [ ]:
# % of time spent in each location
count_location = df['location'].value_counts(normalize=True) * 100

In [ ]:
count_location

In [ ]:
# distribution of the midpoint, sleep onset, sleep offset, and sleep duration
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(df['sleep_start_decimal'].dropna(), kde=True)
plt.title("Distribution of sleep onset")
plt.xlabel("Sleep onset(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
sns.histplot(df['sleep_end_decimal'].dropna(), kde=True)
plt.title("Distribution of wake up time")
plt.xlabel("Wake up time(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
sns.histplot(df['sleep_duration'].dropna(), kde=True)
plt.title("Distribution of sleep duration")
plt.xlabel("sleep duration(h)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# distribution of phase and midpoint of sleep
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(df['phase'].dropna(), kde=True)
plt.title("Distribution of phase(wake up time-sunrise)")
plt.xlabel("Phase(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
sns.histplot(data=df, x='midsleep_h', kde=True, hue='location')
plt.title("Distribution of midsleep")
plt.xlabel("midsleep(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
sns.histplot(data=df, x='waso', kde=True)
plt.title("Distribution of WASO")
plt.xlabel("WASO(min)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Test normality by Q-Q plot
fig, ax = plt.subplots(1, 4, figsize=(16, 5))

sm.qqplot(df['midsleep_h'].dropna(), line='s', ax=ax[0])
ax[0].set_title('Q-Q plot for midsleep')

sm.qqplot(df['sleep_duration'].dropna(), line='s', ax=ax[1])
ax[1].set_title('Q-Q plot for sleep duration')

sm.qqplot(df['phase'].dropna(), line='s', ax=ax[2])
ax[2].set_title('Q-Q plot for phase')

sm.qqplot(df['waso'].dropna(), line='s', ax=ax[3])
ax[3].set_title('Q-Q plot for WASO')

plt.tight_layout()
plt.show()


In [ ]:
# test normality of the data using Shapiro-Wilk test 
# Null hipotesis(H0): data is normally distributed
shapiro_test_sleep_duration = stats.shapiro(df['sleep_duration'])
shapiro_test_midsleep = stats.shapiro(df['midsleep_h'])
shapiro_test_sleep_start = stats.shapiro(df['sleep_start_decimal'])
shapiro_test_sleep_end = stats.shapiro(df['sleep_end_decimal'])
shapiro_test_phase = stats.shapiro(df['phase'])
shapiro_test_waso = stats.shapiro(df['waso'].dropna())

In [ ]:
shapiro_results_x = pd.DataFrame({
    'Variable': ['sleep_duration', 'midsleep_h', 'sleep_start_decimal', 'sleep_end_decimal', 'phase(sleepoffset-sunrise)', 'waso(min)'],
    'Shapiro-Wilk test': [shapiro_test_sleep_duration, shapiro_test_midsleep, shapiro_test_sleep_start, shapiro_test_sleep_end, shapiro_test_phase, shapiro_test_waso]
})

In [ ]:
shapiro_results_x

In [ ]:
# Function to convert hours to hh:mm format
def hours_to_hhmm(x, pos):
    hours = int(x)
    minutes = int((x - hours) * 60)
    return f'{hours:02d}:{minutes:02d}'

__Sleep-wake pattern between workdays and free days__

In [ ]:
# descriptive statistics by location
df_grouped_weekdaytype = df.groupby('weekday_type').describe()

In [ ]:
# compare variables between workdays and free days
# compare the variables between ITA and UK
ttest_midsleep_daytype = stats.ttest_ind(df[df['weekday_type'] == 'work days']['midsleep_h'], df[df['weekday_type'] == 'free days']['midsleep_h'])
ttest_duration_daytype = stats.ttest_ind(df[df['weekday_type'] == 'work days']['sleep_duration'], df[df['weekday_type'] == 'free days']['sleep_duration'])

utest_phase_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['phase'], df[df['weekday_type'] == 'free days']['phase'])
utest_start_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['sleep_start_decimal'], df[df['weekday_type'] == 'free days']['sleep_start_decimal'])
utest_end_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['sleep_end_decimal'], df[df['weekday_type'] == 'free days']['sleep_end_decimal'])
utest_waso_daytype = stats.mannwhitneyu(df[df['weekday_type'] == 'work days']['waso'].dropna(), df[df['weekday_type'] == 'free days']['waso'].dropna())

In [ ]:
# print the results
print('T test results by day type')
print('T test midsleep:', ttest_midsleep_daytype)
print('T test duration:', ttest_duration_daytype)
print('U test results by day type')
print('U test phase:', utest_phase_daytype)
print('U test start:', utest_start_daytype)
print('U test end:', utest_end_daytype)
print('U test waso:', utest_waso_daytype)


__Sleep-wake pattern between Uk and Italy__

In [ ]:
# descriptive statistics by location
df_grouped_location = df.groupby('location').describe()
df_grouped_location = df_grouped_location.transpose()

In [ ]:
# compare the variables between ITA and UK
ttest_midsleep_all_loc = stats.ttest_ind(df[df['location'] == 'ITA']['midsleep_h'], df[df['location'] == 'UK']['midsleep_h'])
ttest_duration_loc = stats.ttest_ind(df[df['location'] == 'ITA']['sleep_duration'], df[df['location'] == 'UK']['sleep_duration'])

utest_phase_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['phase'], df[df['location'] == 'UK']['phase'])
utest_start_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_start_decimal'], df[df['location'] == 'UK']['sleep_start_decimal'])
utest_end_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_end_decimal'], df[df['location'] == 'UK']['sleep_end_decimal'])
utest_waso_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['waso'].dropna(), df[df['location'] == 'UK']['waso'].dropna())

In [ ]:
# print the results
print('T test results by location')
print('Midsleep_all:', ttest_midsleep_all_loc)
print('Sleep_duration:', ttest_duration_loc)
print('U test results by location')
print('Sleep_onset:', utest_start_loc)
print('Sleep_offset:', utest_end_loc)
print('Phase:', utest_phase_loc)
print('WASO:', utest_waso_loc)

In [ ]:
# plot the midpoint of sleep by location
plt.figure(figsize=(6, 6))
sns.boxplot(x='location', y='midsleep_h', data=df)
plt.title('Mean midsleep by location')
plt.suptitle('')  
plt.xlabel('')
plt.ylabel('Midsleep (h)')
plt.ylim(23, 30)
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
# Format y-axis to hh:mm
#plt.gca().yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5)) # gcd stands for 'get current axis'
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.tight_layout()

plt.show()

In [ ]:
# plot the sleep onset and sleep offset by location
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='sleep_start_decimal', data=df, gap=0.2)
plt.title('Mean sleep onset by location')
plt.xlabel('')
plt.ylabel('sleep onset (h)')
plt.annotate('*', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line to the subplot
#plt.ylim(17.5, 27.9)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
#plt.gca().yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))
plt.subplot(1, 2, 2)
sns.boxplot(x='location', y='sleep_end_decimal', data=df, gap=0.2)
plt.title('Mean wake up time by location')
plt.xlabel('')
plt.ylabel('sleep offset (h)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line to the subplot
#plt.ylim(28.5, 36)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
#plt.gca().yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# plot the waso by location
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='waso', data=df, gap=0.2)
plt.title('Mean of Wake After Sleep Onset by location')
plt.xlabel('')
plt.ylabel('WASO (min)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

sns.despine()
plt.tight_layout()
plt.show()

__Jet lag__

In [ ]:
# drop the rows with missing values
jetlag_clean = weekly_means_jetlag['jet lag'].dropna()

In [ ]:
# Test normality of the jet lag data using Shapiro-Wilk test 
# H0: data is normally distributed
shapiro_test_jetlag = stats.shapiro(jetlag_clean)

In [ ]:
print('Shapiro test result for jet lag:')
print(shapiro_test_jetlag)

In [ ]:
# test the difference in jet lag between the two locations
ttest_jetlag = stats.ttest_ind(weekly_means_jetlag[weekly_means_jetlag['location'] == 'ITA']['jet lag'].dropna(), 
                               weekly_means_jetlag[weekly_means_jetlag['location'] == 'UK']['jet lag'].dropna())

In [ ]:
print('T test result for jet lag by location:')
print(ttest_jetlag)

__Season and sleep-wake pattern__

In [ ]:
# remove NaN values from the columns and create a new dataframe
df1 = df.dropna(subset=['sleep_duration']) 
df2 = df.dropna(subset=['phase'])
df3 = df.dropna(subset=['waso'])

In [ ]:
anova_ols_midsleep_season = ols('midsleep_h ~ C(season)', data=df).fit() # generate and fit the regression model
anova_results_midsleep = sm.stats.anova_lm(anova_ols_midsleep_season, typ=3) # fit the ANOVA model and get the results

In [ ]:
print('ANOVA Result for midsleep:')
print(anova_results_midsleep)

In [ ]:
anova_sleep_duration_season = ols('sleep_duration ~ C(season)', data=df1).fit()
anova_results_sleep_duration = sm.stats.anova_lm(anova_sleep_duration_season, typ=3)

print('ANOVA Result for sleep duration:')
print(anova_results_sleep_duration)

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season = pairwise_tukeyhsd(df1['sleep_duration'], df1['season'])
print(tukey_results_season)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='sleep_duration', data=df1)
plt.title('Sleep duration by location')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Sleep duration (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().spines['bottom'].set_color('black') 
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# perform a Kruskal-Wallis test to compare the phase, sleep end and waso between the seasons
kw_phase_season = stats.kruskal(df2[df2['season'] == 'Winter']['phase'], df2[df2['season'] == 'Spring']['phase'], 
                                df2[df2['season'] == 'Summer']['phase'], df2[df2['season'] == 'Autumn']['phase'])

kw_start_season = stats.kruskal(df[df['season'] == 'Winter']['sleep_start_decimal'], df[df['season'] == 'Spring']['sleep_start_decimal'], 
                              df[df['season'] == 'Summer']['sleep_start_decimal'], df[df['season'] == 'Autumn']['sleep_start_decimal'])

kw_end_season = stats.kruskal(df[df['season'] == 'Winter']['sleep_end_decimal'], df[df['season'] == 'Spring']['sleep_end_decimal'], 
                              df[df['season'] == 'Summer']['sleep_end_decimal'], df[df['season'] == 'Autumn']['sleep_end_decimal'])

kw_waso_season = stats.kruskal(df3[df3['season'] == 'Winter']['waso'], df3[df3['season'] == 'Spring']['waso'], 
                               df3[df3['season'] == 'Summer']['waso'], df3[df3['season'] == 'Autumn']['waso'])

print('Kruskal-Wallis test results for phase:')
print(kw_phase_season)
print('Kruskal-Wallis test results for sleep start:')
print(kw_start_season)
print('Kruskal-Wallis test results for sleep end:')
print(kw_end_season)
print('Kruskal-Wallis test results for waso:')  
print(kw_waso_season)

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season = pairwise_tukeyhsd(df2['phase'], df2['season'])
print(tukey_results_season)

In [ ]:
# phase (sleep offset - sunrise) by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='phase', data=df2)
plt.title('Phase (wake up time-sunrise) by season')
plt.suptitle('')  # add space after the title
plt.ylabel('Phase (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season = pairwise_tukeyhsd(df['sleep_start_decimal'], df['season'])
print(tukey_results_season)

In [ ]:
# phase (sleep offset - sunrise) by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='sleep_start_decimal', data=df2)
plt.title('Sleep onset by season')
plt.suptitle('')  # add space after the title
plt.ylabel('Sleep onset (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

__DST and sleep-wake pattern__

In [ ]:
# t-test to compare the midpoint of sleep between DST and non-DST
ttest_midsleep_dst = stats.ttest_ind(df[df['DST_1'] == 0]['midsleep_h'], df[df['DST_1'] == 1]['midsleep_h'])
ttest_sleep_duration_dst = stats.ttest_ind(df1[df1['DST_1'] == 0]['sleep_duration'], df1[df1['DST_1'] == 1]['sleep_duration'])

utest_sleep_start_dst = stats.mannwhitneyu(df[df['DST_1'] == 0]['sleep_start_decimal'], df[df['DST_1'] == 1]['sleep_start_decimal'])
utest_sleep_end_dst = stats.mannwhitneyu(df[df['DST_1'] == 0]['sleep_end_decimal'], df[df['DST_1'] == 1]['sleep_end_decimal'])
utest_phase_dst = stats.mannwhitneyu(df2[df2['DST_1'] == 0]['phase'], df2[df2['DST_1'] == 1]['phase'])
utest_waso_dst = stats.mannwhitneyu(df3[df3['DST_1'] == 0]['waso'], df3[df3['DST_1'] == 1]['waso'])

In [ ]:
print('T test result for the midsleep by DST:')
print(ttest_midsleep_dst)
print('T test result for the sleep duration by DST:')
print(ttest_sleep_duration_dst)
print('U test result for the sleep onset by DST:')
print(utest_sleep_start_dst)
print('U test result for the sleep offset by DST:')
print(utest_sleep_end_dst)
print('U test result for the phase by DST:')
print(utest_phase_dst)
print('U test result for the waso by DST:')
print(utest_waso_dst)

In [ ]:
# mean and standard deviation of the phase by DST
df_grouped_dst = df.groupby('DST_1').agg({'phase': ['mean', 'std']})
df_grouped_dst = df_grouped_dst.reset_index()
df_grouped_dst.columns = ['DST', 'mean', 'std']

df_grouped_dst

In [ ]:
# phase (sleep offset - sunrise) by DST
plt.figure(figsize=(8, 7))
sns.boxplot(x='DST_1', y='phase', data=df2)
plt.title('Phase (wake up time-sunrise) by time shift')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Phase (h)')
plt.xlabel('')
plt.xticks([0, 1], ['ST', 'DST'])
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.annotate('***', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

__Photoperiod and sleep-wake pattern__

In [ ]:
# correlation between sleep-wake variables and photoperiod
correlation1_test1 = stats.pearsonr(df['midsleep_h'], df['photoperiod'])
correlation1_test2 = stats.pearsonr(df_workdays['midsleep_h'], df_workdays['photoperiod'])
correlation1_test3 = stats.pearsonr(df_freedays['midsleep_h'], df_freedays['photoperiod'])
correlation1_test4 = stats.pearsonr(df2['sleep_duration'], df2['photoperiod'])

# correlation between sleep-wake variables and photoperiod using Spearman's rank correlation
correlation1_test5 = stats.spearmanr(df['sleep_start_decimal'], df['photoperiod'])
correlation1_test6 = stats.spearmanr(df['sleep_end_decimal'], df['photoperiod'])
correlation1_test7 = stats.spearmanr(df3['waso'], df3['photoperiod'])

In [ ]:
# extract the coefficients and p-values from the correlation test results
correlation1_coeff = [correlation1_test1.statistic, correlation1_test2.statistic, correlation1_test3.statistic, 
                     correlation1_test4.statistic, correlation1_test5.statistic, correlation1_test6.statistic,
                     correlation1_test7.statistic]

p_values = [correlation1_test1.pvalue, correlation1_test2.pvalue, correlation1_test3.pvalue, 
            correlation1_test4.pvalue, correlation1_test5.pvalue, correlation1_test6.pvalue,
            correlation1_test7.pvalue]

In [ ]:
# create a DataFrame with the results
correlation1_results = pd.DataFrame({
    'Variables': ['midsleep vs photoperiod', 'midsleep (w) vs photoperiod', 'midsleep (f) vs photoperiod', 
                  'sleep duration vs photoperiod', 'sleep onset vs photoperiod', 'sleep offset vs photoperiod', 'WASO(min) vs photoperiod'],
    'Coefficient': correlation1_coeff,
    'P-value': p_values
})

correlation1_results

In [ ]:
# correlation between sleep-wake variables and photoperiod
correlation2_test1 = stats.pearsonr(df['midsleep_h_UTC'], df['photoperiod'])
correlation2_test2 = stats.pearsonr(df_workdays['midsleep_h_UTC'], df_workdays['photoperiod'])
correlation2_test3 = stats.pearsonr(df_freedays['midsleep_h_UTC'], df_freedays['photoperiod'])
correlation2_test4 = stats.pearsonr(df2['sleep_duration'], df2['photoperiod'])

# correlation between sleep-wake variables and photoperiod using Spearman's rank correlation
correlation2_test5 = stats.spearmanr(df['sleep_start_decimal_UTC'], df['photoperiod'])
correlation2_test6 = stats.spearmanr(df['sleep_end_decimal_UTC'], df['photoperiod'])
correlation2_test7 = stats.spearmanr(df3['waso'], df3['photoperiod'])

In [ ]:
# extract the coefficients and p-values from the correlation test results
correlation2_coeff = [correlation2_test1.statistic, correlation2_test2.statistic, correlation2_test3.statistic, 
                     correlation2_test4.statistic, correlation2_test5.statistic, correlation2_test6.statistic,
                     correlation2_test7.statistic]

p_values = [correlation2_test1.pvalue, correlation2_test2.pvalue, correlation2_test3.pvalue, 
            correlation2_test4.pvalue, correlation2_test5.pvalue, correlation2_test6.pvalue,
            correlation2_test7.pvalue]

In [ ]:
# create a DataFrame with the results
correlation2_results = pd.DataFrame({
    'Variables': ['midsleep vs photoperiod', 'midsleep (w) vs photoperiod', 'midsleep (f) vs photoperiod', 
                  'sleep duration vs photoperiod', 'sleep onset vs photoperiod', 'sleep offset vs photoperiod', 'WASO(min) vs photoperiod'],
    'Coefficient': correlation2_coeff,
    'P-value': p_values
})

correlation2_results

In [ ]:
# plot the correlation between the midpoint of sleep and photoperiod, for all the data, work days and free days
plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
sns.scatterplot(x='midsleep_h_UTC', y='photoperiod', data=df)
sns.regplot(x='midsleep_h_UTC', y='photoperiod', data=df, scatter=False, color='red')
corr_all, _ = stats.pearsonr(df['midsleep_h_UTC'], df['photoperiod'])
plt.title(f'Midsleep vs photoperiod\nCorrelation: {corr_all:.2f}')
plt.xlabel('Midsleep (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(5, 20)

plt.subplot(1, 3, 2)
sns.scatterplot(x='sleep_start_decimal_UTC', y='photoperiod', data=df)
sns.regplot(x='sleep_start_decimal_UTC', y='photoperiod', data=df, scatter=False, color='red')
corr_free, _ = stats.pearsonr(df['sleep_start_decimal_UTC'], df['photoperiod'])
plt.title(f'Sleep onset vs photoperiod\nCorrelation: {corr_free:.2f}')
plt.xlabel('Sleep onset (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(5, 20)

plt.subplot(1, 3, 3)
sns.scatterplot(x='sleep_end_decimal_UTC', y='photoperiod', data=df)
sns.regplot(x='sleep_end_decimal_UTC', y='photoperiod', data=df, scatter=False, color='red')
corr_offset, _ = stats.pearsonr(df['sleep_end_decimal_UTC'], df['photoperiod'])
plt.title(f'Sleep offset vs photoperiod\nCorrelation: {corr_offset:.2f}')
plt.xlabel('Sleep offset (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(5, 20)

plt.tight_layout()
plt.show()

In [ ]:
# distribution of the photoperiod
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='photoperiod', kde=False, hue='location', multiple='dodge')
plt.title('Distribution of photoperiod')
plt.xlabel('Photoperiod (h)')
plt.ylabel('Frequency')

# Add vertical lines for the tertiles
tertiles = df['photoperiod'].quantile([1/3, 2/3])
for tertile in tertiles:
    plt.axvline(tertile, color='r', linestyle='--')

plt.xlim(5, 20)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.show()

In [ ]:
# define the tertile of the photoperiod
df['photoperiod_tertile'] = pd.qcut(df['photoperiod'], 3, labels=['short', 'medium', 'long'])
df2['photoperiod_tertile'] = pd.qcut(df2['photoperiod'], 3, labels=['short', 'medium', 'long'])
df3['photoperiod_tertile'] = pd.qcut(df3['photoperiod'], 3, labels=['short', 'medium', 'long'])

In [ ]:
# print the threshold for the tertiles
print('Tertiles for the photoperiod:')
print(tertiles)

In [ ]:
# what is photoperiod terile
df_grouped_tertile = df.groupby('photoperiod_tertile').describe()

df_grouped_tertile = df_grouped_tertile.transpose()
df_grouped_tertile = df_grouped_tertile.reset_index()

In [ ]:
#add a column in df, df1, df2, df3 with the photoperiod tertile
df['photoperiod_tertile'] = pd.qcut(df['photoperiod'], 3, labels=['short', 'medium', 'long'])
df1['photoperiod_tertile'] = pd.qcut(df1['photoperiod'], 3, labels=['short', 'medium', 'long'])
df2['photoperiod_tertile'] = pd.qcut(df2['photoperiod'], 3, labels=['short', 'medium', 'long'])
df3['photoperiod_tertile'] = pd.qcut(df3['photoperiod'], 3, labels=['short', 'medium', 'long'])

In [ ]:
#compare the variables between the photoperiod tertiles
ttest_midsleep_tertile = stats.f_oneway(df[df['photoperiod_tertile'] == 'short']['midsleep_h'], 
                                        df[df['photoperiod_tertile'] == 'medium']['midsleep_h'], 
                                        df[df['photoperiod_tertile'] == 'long']['midsleep_h'])

ttest_duration_tertile = stats.f_oneway(df1[df1['photoperiod_tertile'] == 'short']['sleep_duration'],
                                        df1[df1['photoperiod_tertile'] == 'medium']['sleep_duration'],
                                        df1[df1['photoperiod_tertile'] == 'long']['sleep_duration'])

utest_phase_tertile = stats.kruskal(df2[df2['photoperiod_tertile'] == 'short']['phase'],
                                    df2[df2['photoperiod_tertile'] == 'medium']['phase'],
                                    df2[df2['photoperiod_tertile'] == 'long']['phase'])

utest_start_tertile = stats.kruskal(df[df['photoperiod_tertile'] == 'short']['sleep_start_decimal'],
                                    df[df['photoperiod_tertile'] == 'medium']['sleep_start_decimal'],
                                    df[df['photoperiod_tertile'] == 'long']['sleep_start_decimal'])

utest_end_tertile = stats.kruskal(df[df['photoperiod_tertile'] == 'short']['sleep_end_decimal'],
                                    df[df['photoperiod_tertile'] == 'medium']['sleep_end_decimal'],
                                    df[df['photoperiod_tertile'] == 'long']['sleep_end_decimal'])

utest_waso_tertile = stats.kruskal(df3[df3['photoperiod_tertile'] == 'short']['waso'],
                                    df3[df3['photoperiod_tertile'] == 'medium']['waso'],
                                    df3[df3['photoperiod_tertile'] == 'long']['waso'])

In [ ]:
print('ANOVA results for midsleep by photoperiod tertile:')
print(ttest_midsleep_tertile)
print('ANOVA results for sleep duration by photoperiod tertile:')
print(ttest_duration_tertile)
print('Kruskal-Wallis results for phase by photoperiod tertile:')
print(utest_phase_tertile)
print('Kruskal-Wallis results for sleep start by photoperiod tertile:')
print(utest_start_tertile)
print('Kruskal-Wallis results for sleep end by photoperiod tertile:')
print(utest_end_tertile)
print('Kruskal-Wallis results for waso by photoperiod tertile:')
print(utest_waso_tertile)

In [ ]:
# define different df for the 3 photoperiod tertiles
df_short = df[df['photoperiod_tertile'] == 'short']
df_medium = df[df['photoperiod_tertile'] == 'medium']
df_long = df[df['photoperiod_tertile'] == 'long']

In [ ]:
# ttest between Ita and UK photoperiod in shorter photoperiod
ttest_midsleep_short = stats.ttest_ind(df_short[df_short['location'] == 'ITA']['midsleep_h'], df_short[df_short['location'] == 'UK']['midsleep_h'])
ttest_duration_short = stats.ttest_ind(df_short[df_short['location'] == 'ITA']['sleep_duration'], df_short[df_short['location'] == 'UK']['sleep_duration'])

utest_phase_short = stats.mannwhitneyu(df_short[df_short['location'] == 'ITA']['phase'], df_short[df_short['location'] == 'UK']['phase'])
utest_start_short = stats.mannwhitneyu(df_short[df_short['location'] == 'ITA']['sleep_start_decimal'], df_short[df_short['location'] == 'UK']['sleep_start_decimal'])
utest_end_short = stats.mannwhitneyu(df_short[df_short['location'] == 'ITA']['sleep_end_decimal'], df_short[df_short['location'] == 'UK']['sleep_end_decimal'])
utest_waso_short = stats.mannwhitneyu(df_short[df_short['location'] == 'ITA']['waso'].dropna(), df_short[df_short['location'] == 'UK']['waso'].dropna())

In [ ]:
# print the results
print('T test results by location for shorter photoperiod')
print('Midsleep:', ttest_midsleep_short)
print('Sleep duration:', ttest_duration_short)
print('U test results by location for shorter photoperiod')
print('Phase:', utest_phase_short)
print('Sleep onset:', utest_start_short)
print('Sleep offset:', utest_end_short)
print('WASO:', utest_waso_short)

In [ ]:
# test between Ita and UK photoperiod in longer photoperiod
ttest_long_midsleep = stats.ttest_ind(df_long[df_long['location'] == 'ITA']['midsleep_h'], df_long[df_long['location'] == 'UK']['midsleep_h'])
ttest_long_sleep_duration = stats.ttest_ind(df_long[df_long['location'] == 'ITA']['sleep_duration'], df_long[df_long['location'] == 'UK']['sleep_duration'])

utest_long_phase = stats.mannwhitneyu(df_long[df_long['location'] == 'ITA']['phase'], df_long[df_long['location'] == 'UK']['phase'])
utest_long_start = stats.mannwhitneyu(df_long[df_long['location'] == 'ITA']['sleep_start_decimal'], df_long[df_long['location'] == 'UK']['sleep_start_decimal'])
utest_long_end = stats.mannwhitneyu(df_long[df_long['location'] == 'ITA']['sleep_end_decimal'], df_long[df_long['location'] == 'UK']['sleep_end_decimal'])
utest_long_waso = stats.mannwhitneyu(df_long[df_long['location'] == 'ITA']['waso'].dropna(), df_long[df_long['location'] == 'UK']['waso'].dropna())

In [ ]:
#print the results
print('T test results for longer photoperiod:')
print('T test midsleep:', ttest_long_midsleep)
print('T test sleep duration:', ttest_long_sleep_duration)
print('U test results for longer photoperiod:')
print('U test phase:', utest_long_phase)
print('U test sleep onset:', utest_long_start)
print('U test sleep offset:', utest_long_end)
print('U test waso:', utest_long_waso)

In [ ]:
# bonferroni correction
alpha = 0.05
n_tests = 6
alpha_corrected = alpha / n_tests

alpha_corrected

In [ ]:
# mean and standard deviation of the sleep var by photoperiod tertile and location
midsleep_phot_loc = df.groupby(['photoperiod_tertile', 'location']).agg({'sleep_end_decimal': ['mean', 'std'], 'sleep_start_decimal': ['mean', 'std'], 
                                                                             'phase': ['mean', 'std'], 'waso': ['mean', 'std']})
midsleep_phot_loc = midsleep_phot_loc.reset_index()
midsleep_phot_loc.columns = ['Photoperiod', 'Location', 'Mean wake up time', 'Std wake up time', 'Mean sleep onset', 'Std sleep onset', 
                                  'Mean phase', 'Std phase', 'Mean WASO', 'Std WASO']

midsleep_phot_loc

In [ ]:
plt.figure(figsize=(8, 7))
sns.boxplot(x='location', y='sleep_end_decimal', data=df_short)
plt.title('Sleep offset by location for shorter photoperiod')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Sleep offset (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.ylim(28, 36)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))
sns.boxplot(x='location', y='sleep_start_decimal', data=df_long)
plt.title('Sleep onset by location for longer photoperiod')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Sleep onset (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))
sns.boxplot(x='location', y='waso', data=df_long)
plt.title('Wake After Sleep Onset by location for longer photoperiod')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('WASO (min)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.annotate('**', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))
sns.boxplot(x='location', y='phase', data=df_long)
plt.title('Phase by location for longer photoperiod')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Phase (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

__Weekly IV, IS and RA__

In [ ]:
# load the datasets required for the analysis
weekly_values = pd.read_csv(fpath + '\\2.0_weekly_IV_IS_RA_values_with_dates.csv')
weekly_jetlag = weekly_means_jetlag

In [ ]:
# split Date_Range into Start_Date and End_Date in weekly_values
weekly_values[['Start_Date', 'End_Date']] = weekly_values['Date_Range'].str.split(' to ', expand=True)

In [ ]:
# convert Start_Date and End_Date to datetime format
weekly_values['Start_Date'] = pd.to_datetime(weekly_values['Start_Date'])
weekly_values['End_Date'] = pd.to_datetime(weekly_values['End_Date'])

In [ ]:
# merge by matching the week number extracted from Start_Date with week_of_year in weekly_jetlag
merged_data = pd.merge(
    weekly_values,
    weekly_jetlag,
    left_on=weekly_values['Start_Date'].dt.isocalendar().week,
    right_on='week_of_year',
    how='inner'
)

In [ ]:
merged_data.head()

In [ ]:
# box plot to verify the outliers in IV, IS, and RA
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
sns.boxplot(data=merged_data['IV'], ax=ax[0])
sns.boxplot(data=merged_data['IS'], ax=ax[1])
sns.boxplot(data=merged_data['RA'], ax=ax[2])
plt.show()

In [ ]:
# summary statistics
summary_stats = merged_data.groupby("location")[['IV', 'IS', 'RA']].describe()

summary_stats

In [ ]:
# distribution of IV, IS, and RA
plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
sns.histplot(merged_data['IV'].dropna(), kde=True)
plt.title('Distribution of IV')
plt.xlabel('IV')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
sns.histplot(merged_data['IS'].dropna(), kde=True)
plt.title('Distribution of IS')
plt.xlabel('IS')
plt.ylabel('Frequency')
 
plt.subplot(1, 3, 3)
sns.histplot(merged_data['RA'].dropna(), kde=True)
plt.title('Distribution of RA')
plt.xlabel('RA')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# compare the variables between ITA and UK
iv_ttest = stats.ttest_ind(merged_data[merged_data['location'] == 'ITA']['IV'], merged_data[merged_data['location'] == 'UK']['IV'], nan_policy='omit')
is_utest = stats.mannwhitneyu(merged_data[merged_data['location'] == 'ITA']['IS'], merged_data[merged_data['location'] == 'UK']['IS'], nan_policy='omit')
ra_utest = stats.mannwhitneyu(merged_data[merged_data['location'] == 'ITA']['RA'], merged_data[merged_data['location'] == 'UK']['RA'], nan_policy='omit')

In [ ]:
print('Test results for IV by location:', iv_ttest)
print('Test results for IS by location:', is_utest)
print('Test results for RA by location:', ra_utest)

------------------------

__Sleep-wake patterns over time__

In [ ]:
# filter df to have only day_after_flight 1,2,8,9
df_flight1 = df[df['day_after_flight'].isin([1, 2, 8, 9])]

In [ ]:
# new column with the day after flight day 1 and 2 as '0' and day 8 and 9 as '1'
df_flight1['day_after_flight_group(0=day1&2;1=day8&9)'] = np.where(df_flight1['day_after_flight'].isin([1, 2]), '0', '1')

In [ ]:
# rename df_flight1['day_after_flight_group(0=day1&2;1=day8&9)']
df_flight1 = df_flight1.rename(columns={'day_after_flight_group(0=day1&2;1=day8&9)': 'day_after_flight_group'})

In [ ]:
# remove outliers
# criteria: zscore of 3 means that the data point is 3 standard deviations away from the mean
df = df[(np.abs(stats.zscore(df['midsleep_h'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_start_decimal'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_end_decimal'])) < 3)]
df = df[(np.abs(stats.zscore(df['phase'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_duration'])) < 3)]

In [ ]:
# filter df to have only day_after_flight > 15
#df = df[df['day_after_flight'] > 15] 

_Sleep onset_

In [ ]:
model1 = smf.mixedlm('sleep_start_decimal ~ day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000)

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('sleep_start_decimal ~ day_after_flight + location + location*day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
# shapiro-wilk test for sleep_start_decimal
shapiro_test_sleep_start = stats.shapiro(df_flight1['sleep_start_decimal'])

shapiro_test_sleep_start

In [ ]:
ttest_start_days = stats.ttest_ind(df_flight1[df_flight1['day_after_flight_group'] == '0']['sleep_start_decimal'], df_flight1[df_flight1['day_after_flight_group'] == '1']['sleep_start_decimal'])

ttest_start_days

In [ ]:
# mean and std of sleep_start_decimal by day_after_flight_group 
df_grouped_flight = df_flight1.groupby(['day_after_flight_group']).agg({'sleep_start_decimal': ['mean', 'std']})
df_grouped_flight


In [ ]:
# plot the sleep onset by day after flight
plt.figure(figsize=(6, 6))
sns.boxplot(x='day_after_flight_group', y='sleep_start_decimal', data=df_flight1)
plt.title('Sleep onset day 1-2 VS day 8-9 after each flight')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Sleep onset (h)')
plt.xlabel('')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(18.5, 27)
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.tight_layout()
plt.xticks([0, 1], ['Day 1-2', 'Day 8-9'])

plt.show()

In [ ]:
# Fit the factorial ANOVA model
model4 = ols('sleep_start_decimal ~ C(location) * C(day_after_flight_group)', data=df_flight1).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model4, typ=2)

print(anova_results)

In [ ]:
# Fit a mixed-effects model with random intercepts and slopes for location and flight_id
model5 = smf.mixedlm('sleep_start_decimal ~ C(location) + photoperiod + C(location)*photoperiod + day_after_flight + C(location)*day_after_flight',
                      data=df, groups=df['flight_id']).fit(method='powell', maxiter=1000)

print(model5.summary())

In [ ]:
coef_onset = pd.DataFrame({'coef': model5.params.values, 'p-value': model5.pvalues.values, '0.025': model5.conf_int()[0], '0.975': model5.conf_int()[1]})
coef_onset

In [ ]:
#drop non significant variables
coef_onset = coef_onset.drop('Intercept')
coef_onset = coef_onset.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_onset.index, data=coef_onset, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_onset.shape[0]):
    plt.plot([coef_onset['0.025'].iloc[i], coef_onset['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_onset['0.025'].iloc[i], coef_onset['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_onset['0.975'].iloc[i], coef_onset['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_onset.shape[0]):
    if coef_onset['p-value'].iloc[i] < 0.001:
        plt.text(coef_onset['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_onset['p-value'].iloc[i] < 0.01:
        plt.text(coef_onset['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_onset['p-value'].iloc[i] < 0.06:
        plt.text(coef_onset['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=sleep onset)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.89)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location [T.UK]', 'Photoperiod', 'Location [T.UK]*Photoperiod','Days after flight', 'Location [T.UK]*Days after flight'], fontsize=9)

plt.show()

In [ ]:
# QQ-plot to verify the residuals of the model
plt.figure(figsize=(8, 6))
sm.qqplot(model5.resid, line='45')
plt.title('QQ-plot of the residuals')
plt.show()

In [ ]:
residualsY = model5.resid 
predictedY = model5.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_testY = het_breuschpagan(residualsY, model5.model.exog)

# results of the Breusch-Pagan test
bp_statY, bp_pvalY, _, _ = bp_testY
print(f'Breusch-Pagan statistic: {bp_statY}, p-value: {bp_pvalY}')
if bp_pvalY > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_testY = durbin_watson(residualsY)

print('Durbin-Watson test:', durbin_watson_testY)

In [ ]:
# The Durbin-Watson test statistic is close to 2, which indicates that there is no significant autocorrelation in the residuals

In [ ]:
#extract from df sunrise and sunset times for both the UK and Italy and create a new dataframe df_daylight
df_daylight1 = pd.read_excel(fpath + '\\1.0_sunset_sunrise_UTC.xlsx', sheet_name='Sheet1')

In [ ]:
def adjust_value(row):
    timeshift = row['DST(0=ST)']
    
    if timeshift == 1:
                return row['sunrise (uk), hours'] + 1, row['sunset (uk), hours'] + 1, row['sunrise (ita), hours'] + 2, row['sunset (ita), hours'] + 2
    elif timeshift == 0:
                return row['sunrise (uk), hours'] - 0, row['sunset (uk), hours'] - 0, row['sunrise (ita), hours'] + 1, row['sunset (ita), hours'] + 1
    
    return row['sunrise (uk), hours'], row['sunset (uk), hours'], row['sunrise (ita), hours'], row['sunset (ita), hours']

df_daylight1[['sunrise (uk), hours_adjust', 'sunset (uk), hours_adjust', 'sunrise (ita), hours_adjust', 'sunset (ita), hours_adjust']] = df_daylight1.apply(adjust_value, axis=1, result_type='expand')

In [ ]:
#add 24 hours to the sleep_start_decimal and sleep_end_decimal
df['sleep_end_decimal_plot'] = df['sleep_end_decimal'] - 24
df['sleep_end_decimal_UTC_plot'] = df['sleep_end_decimal_UTC'] - 24

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# UK sunrise and sunset
sns.lineplot(x='date', y='sunrise (uk), hours_adjust', data=df_daylight1, label='Sunrise UK', color='#675300', ax=ax1)
sns.lineplot(x='date', y='sunset (uk), hours_adjust', data=df_daylight1, label='Sunset UK', color='#b578ff', ax=ax1)

# ITA sunrise and sunset
sns.lineplot(x='date', y='sunrise (ita), hours_adjust', data=df_daylight1, label='Sunrise ITA', color='#fd2f60', ax=ax1)
sns.lineplot(x='date', y='sunset (ita), hours_adjust', data=df_daylight1, label='Sunset ITA', color='#c530b4', ax=ax1)

# sleep onset and offset
sns.scatterplot(x='date', y='sleep_start_decimal', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)
sns.scatterplot(x='date', y='sleep_end_decimal_plot', hue='location', data=df, style='weekday_type', palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Add a third y-axis for daylight length
ax2 = ax1.twinx()
sns.lineplot(x='date', y=df_daylight1['sunset (uk), hours_adjust'] - df_daylight1['sunrise (uk), hours_adjust'], data=df_daylight1, ax=ax2, label='Daylength UK', color='green', linestyle='--')
sns.lineplot(x='date', y=df_daylight1['sunset (ita), hours_adjust'] - df_daylight1['sunrise (ita), hours_adjust'], data=df_daylight1, ax=ax2, label='Daylength ITA', color='lime', linestyle='--')

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Time of Day (hh:mm, local time)')
ax1.set_title('Sleep wake timing in UK and Italy over time', fontsize=16)
ax2.set_ylabel('Daylength (hours)')

# Remove space y axis and plot
plt.gca().margins(x=0)

# Format y-axis to hh:mm
ax1.yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))

# 3 hours interval on y-axis
ax1.set_yticks(range(0, 30, 1))
ax1.set_yticklabels([hours_to_hhmm(i % 24, None) for i in range(0, 30, 1)])

#set ax2 y interval
ax2.set_yticks(range(8, 17, 1))

# Remove space y axis and plot
ax1.margins(x=0)

# Adding legend to the right of the plot
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
by_label = dict(zip(labels1 + labels2, handles1 + handles2))
ax1.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.get_legend().remove()

# Set x-axis major locator of ax1
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start ST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

In [ ]:
# Add a third y-axis for daylight length
fig, ax1 = plt.subplots(figsize=(25, 8))

# Remove spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add gray background
ax1.set_facecolor('#f0f0f0')

# UK sunrise and sunset
sns.lineplot(x='date', y='sunrise (uk), hours', data=df_daylight1, label='Sunrise UK', color='#675300', ax=ax1)
sns.lineplot(x='date', y='sunset (uk), hours', data=df_daylight1, label='Sunset UK', color='#b578ff', ax=ax1)

# ITA sunrise and sunset
sns.lineplot(x='date', y='sunrise (ita), hours', data=df_daylight1, label='Sunrise ITA', color='#fd2f60', ax=ax1)
sns.lineplot(x='date', y='sunset (ita), hours', data=df_daylight1, label='Sunset ITA', color='#c530b4', ax=ax1)

# sleep onset and offset
sns.scatterplot(x='date', y='sleep_start_decimal_UTC', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)
sns.scatterplot(x='date', y='sleep_end_decimal_UTC_plot', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'], legend='full', ax=ax1)

# Add a third y-axis for daylight length
ax2 = ax1.twinx()
sns.lineplot(x='date', y=df_daylight1['sunset (uk), hours'] - df_daylight1['sunrise (uk), hours'], data=df_daylight1, ax=ax2, label='Daylength UK', color='green', linestyle='--')
sns.lineplot(x='date', y=df_daylight1['sunset (ita), hours'] - df_daylight1['sunrise (ita), hours'], data=df_daylight1, ax=ax2, label='Daylength ITA', color='lime', linestyle='--')

# Adding labels and title
ax1.set_xlabel('')
ax1.set_ylabel('Time of Day (hh:mm, UTC)')
ax1.set_title('Sleep wake timing in UK and Italy over time', fontsize=16)
ax2.set_ylabel('Daylength (hours)')

# Remove space y axis and plot
plt.gca().margins(x=0)

# Format y-axis to hh:mm
ax1.yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))

# 3 hours interval on y-axis
ax1.set_yticks(range(0, 30, 1))
ax1.set_yticklabels([hours_to_hhmm(i % 24, None) for i in range(0, 30, 1)])

#set ax2 y interval
ax2.set_yticks(range(8, 17, 1))

# Remove space y axis and plot
ax1.margins(x=0)

# Adding legend to the right of the plot
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
by_label = dict(zip(labels1 + labels2, handles1 + handles2))
ax1.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left')
ax2.get_legend().remove()

# Set x-axis major locator of ax1
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Add vertical lines and text for DST and ST
for date, label in [('2022-10-30', 'start ST'), ('2023-10-29', 'start ST'), ('2023-03-26', 'start DST'), ('2024-03-31', 'start DST'), ('2024-10-27', 'start DST')]:
    plt.axvline(x=pd.to_datetime(date), color='dimgrey', linestyle='--')
    plt.text(pd.to_datetime(date), plt.ylim()[0], label, ha='right', va='bottom', color='dimgray')

plt.xlim(pd.to_datetime('2022-09-19'), pd.to_datetime('2025-01-01'))

plt.show()

_Midsleep_

In [ ]:
# Order the data by date
df = df.sort_values(by='date')
df_flight1 = df_flight1.sort_values(by='date')

In [ ]:
model1 = smf.mixedlm('midsleep_h ~ day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('midsleep_h ~ day_after_flight + location + location*day_after_flight', df, groups=df['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
shapiro_test_midsleep = stats.shapiro(df_flight1['midsleep_h'])

shapiro_test_midsleep

In [ ]:
ttest_midsleep_days = stats.ttest_ind(df_flight1[df_flight1['day_after_flight_group'] == '0']['midsleep_h'], df_flight1[df_flight1['day_after_flight_group'] == '1']['midsleep_h'])

ttest_midsleep_days

In [ ]:
# mean and std of midsleep by day_after_flight_group 
df_grouped_flight = df_flight1.groupby(['day_after_flight_group']).agg({'midsleep_h': ['mean', 'std']})
df_grouped_flight

In [ ]:
# plot the sleep onset by day after flight
plt.figure(figsize=(6, 6))
sns.boxplot(x='day_after_flight_group', y='midsleep_h', data=df_flight1)
plt.title('Midsleep day 1-2 VS day 8-9 after each flight')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Midsleep (h)')
plt.xlabel('')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.ylim(23, 32)
plt.annotate('*', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.tight_layout()
plt.xticks([0, 1], ['Day 1-2', 'Day 8-9'])

plt.show()

In [ ]:
# Fit the factorial ANOVA model
model4 = ols('midsleep_h ~ C(location) * C(day_after_flight_group)', data=df_flight1).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model4, typ=2)

print(anova_results)

In [ ]:
# Fit a mixed-effects model with random intercepts for each date and day after flight
model5 = smf.mixedlm('midsleep_h ~ C(location) + day_after_flight + photoperiod + photoperiod*C(location) + C(location)*day_after_flight', 
                      data=df, groups=df['flight_id']).fit(method='powell', maxiter=1000)

print(model5.summary())

In [ ]:
# calculate log-likelihood of model2a
#ll_model5 = model5.llf
#ll_model5 = model5.llf

# calculate likelihood ratio Chi-Squared test statistic
#lr_test1 = 2 * (ll_model5 - model5.llf)

# calculate p-value of test statistic using 2 degrees of freedom
# p-value > 0.05 means the two models fit the data equally well
#p_value = stats.chi2.sf(lr_test1, 2)

#print('Likelihood ratio test results:')
#print('Chi-Squared test statistic:', lr_test1)
#print('P-value:', p_value)

In [ ]:
coef_midsleep = pd.DataFrame({'coef': model5.params.values, 'p-value': model5.pvalues.values, '0.025': model5.conf_int()[0], '0.975': model5.conf_int()[1]})
coef_midsleep

In [ ]:
#drop non significant variables
coef_midsleep = coef_midsleep.drop('Intercept')
coef_midsleep = coef_midsleep.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_midsleep.index, data=coef_midsleep, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_midsleep.shape[0]):
    plt.plot([coef_midsleep['0.025'].iloc[i], coef_midsleep['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_midsleep['0.025'].iloc[i], coef_midsleep['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_midsleep['0.975'].iloc[i], coef_midsleep['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_midsleep.shape[0]):
    if coef_midsleep['p-value'].iloc[i] < 0.001:
        plt.text(coef_midsleep['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_midsleep['p-value'].iloc[i] < 0.01:
        plt.text(coef_midsleep['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_midsleep['p-value'].iloc[i] < 0.06:
        plt.text(coef_midsleep['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=midsleep)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location [T.UK]', 'Days after flight', 'Location[T.UK]*Days after flight', 'Photoperiod', 'Photoperiod*Location [T.UK]',], fontsize=9)

plt.show()

In [ ]:
# QQ-plot to verify the residuals of the model
plt.figure(figsize=(8, 6))
sm.qqplot(model5.resid, line='45')
plt.title('QQ-plot of the residuals')
plt.show()

In [ ]:
residualsY = model5.resid 
predictedY = model5.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_testY = het_breuschpagan(residualsY, model5.model.exog)

# results of the Breusch-Pagan test
bp_statY, bp_pvalY, _, _ = bp_testY
print(f'Breusch-Pagan statistic: {bp_statY}, p-value: {bp_pvalY}')
if bp_pvalY > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_testY = durbin_watson(residualsY)

print('Durbin-Watson test:', durbin_watson_testY)

In [ ]:
# The Durbin-Watson test statistic is close to 2, which indicates that there is no significant autocorrelation in the residuals

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='midsleep_h', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Midsleep by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h)', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.ylim(23,30)

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start ST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.052, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.05, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.049, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.049, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

# mean of the phase for the two locations
#plt.axhline(y=df[df['location'] == 'ITA']['phase'].mean(), color='darkblue', linestyle='--', label='Mean ITA')
#plt.text(plt.xlim()[1], df[df['location'] == 'ITA']['phase'].mean(), 'Mean ITA', ha='right', va='bottom')
#plt.axhline(y=df[df['location'] == 'UK']['phase'].mean(), color='darkorange', linestyle='--', label='Mean UK')
#plt.text(plt.xlim()[1], df[df['location'] == 'UK']['phase'].mean(), 'Mean UK', ha='right', va='bottom')

#plt.xlim(pd.to_datetime('2024-03-28'), pd.to_datetime('2024-04-27'))

plt.show()

_Phase (wake up time - sunrise)_

In [ ]:
df2 = df
df2_x = df_flight1

In [ ]:
# test the skewness of the phase data
skewness = skew(df2['phase'])

print(f"Index of skewness: {skewness}")

In [ ]:
# test the kurtosis of the phase data
kurtosis_value = kurtosis(df2['phase'])

print(f"Index of kurtosis: {kurtosis_value}")

In [ ]:
#phase_data = df2['phase'].values.reshape(-1, 1)

# apply the Yeo-Johnson transformation
#pt = PowerTransformer(method='yeo-johnson')
#phase_transformed = pt.fit_transform(phase_data)

In [ ]:
# add the transformed phase to the dataframe 
#df2['phase_transformed'] = phase_transformed

In [ ]:
# test the best distribution for the phase data
distributions = ['norm', 'gamma', 'lognorm', 'expon']
best_fit_results = {}

In [ ]:
# Filter out invalid values (e.g., negative values or zeros) for distributions that require positive values
valid_phase = df2["phase"][df2["phase"] > 0]

for dist_name in distributions:
    dist = getattr(stats, dist_name)
    if dist_name in ['gamma', 'lognorm', 'expon']:
        params = dist.fit(valid_phase)
        ks_stat, ks_pval = stats.kstest(valid_phase, dist_name, args=params)
    else:
        params = dist.fit(df2["phase"])
        ks_stat, ks_pval = stats.kstest(df2["phase"], dist_name, args=params)
    best_fit_results[dist_name] = ks_stat  # save the KS statistic

In [ ]:
# plot of the best fit results
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(df2["phase"], bins=30, kde=True, stat="density", label="Data", ax=ax)

x = np.linspace(df2["phase"].min(), df2["phase"].max(), 1000)

# Disegniamo le distribuzioni teoriche
for dist_name in best_fit_results.keys():
    dist = getattr(stats, dist_name)
    params = dist.fit(df2["phase"])
    pdf = dist.pdf(x, *params)
    ax.plot(x, pdf, label=f"{dist_name}")

ax.legend()
ax.set_title("Confronto tra la distribuzione della variabile e distribuzioni teoriche")
plt.show()

In [ ]:
# Shapiro-Wilk test for the transformed phase
shapiro_test_phase = stats.shapiro(df2['phase'])
shapiro_test_phase

In [ ]:
model1 = smf.mixedlm('phase ~ day_after_flight', df2, groups=df2['flight_id']).fit(method='powell')

print(model1.summary())

In [ ]:
model2 = smf.mixedlm('phase ~ day_after_flight + C(location) + C(location)*day_after_flight', df2, groups=df2['flight_id']).fit(method='powell')

print(model2.summary())

In [ ]:
shapiro_test_midsleep = stats.shapiro(df2_x['phase'])

shapiro_test_midsleep

In [ ]:
utest_phase_days = stats.mannwhitneyu(df2_x[df2_x['day_after_flight_group'] == '0']['phase'], df2_x[df2_x['day_after_flight_group'] == '1']['phase'])

utest_phase_days

In [ ]:
# Fit the factorial ANOVA model
model4 = ols('phase ~ C(location) * C(day_after_flight_group)', data=df2_x).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model4, typ=2)

print(anova_results)

In [ ]:
# Fit a mixed-effects model with random intercepts for each day after flight
# re_formula="~1": This specifies that the random effects are independent and only include a random intercept for each group 
# i.e. each group has its own intercept, but the slopes are assumed to be the same across groups
model5a = smf.mixedlm('phase ~ C(location) + photoperiod + C(DST_1) + C(location)*photoperiod', data=df2, groups=df2['flight_id'], re_formula='~1').fit(method='powell', maxiter=1000)

print(model5a.summary())

In [ ]:
# Fit a mixed-effects model with random intercepts for each date and day after flight
model5b = smf.mixedlm('phase ~ C(location) + photoperiod + C(DST_1) + day_after_flight + photoperiod*C(location) + C(location)*day_after_flight', data=df2, groups=df2['flight_id']).fit(method='powell', maxiter=1000)

print(model5b.summary())

In [ ]:
# calculate log-likelihood of model2a
ll_model5a = model5a.llf
ll_model5b = model5b.llf

# calculate likelihood ratio Chi-Squared test statistic
lr_test1 = 2 * (ll_model5a - model5b.llf)

# calculate p-value of test statistic using 2 degrees of freedom
# p-value > 0.05 means the two models fit the data equally well
p_value = stats.chi2.sf(lr_test1, 2)

print('Likelihood ratio test results:')
print('Chi-Squared test statistic:', lr_test1)
print('P-value:', p_value)

In [ ]:
coef_phase = pd.DataFrame({'coef': model5a.params.values, 'p-value': model5a.pvalues.values, '0.025': model5a.conf_int()[0], '0.975': model5a.conf_int()[1]})
coef_phase

In [ ]:
#drop non significant variables
coef_phase = coef_phase.drop('Intercept')
coef_phase = coef_phase.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_phase.index, data=coef_phase, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_phase.shape[0]):
    plt.plot([coef_phase['0.025'].iloc[i], coef_phase['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_phase['0.025'].iloc[i], coef_phase['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_phase['0.975'].iloc[i], coef_phase['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_phase.shape[0]):
    if coef_phase['p-value'].iloc[i] < 0.001:
        plt.text(coef_phase['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_phase['p-value'].iloc[i] < 0.01:
        plt.text(coef_phase['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_phase['p-value'].iloc[i] < 0.06:
        plt.text(coef_phase['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=phase)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location[T.UK]', 'Time shift[T.DST]', 'Photoperiod', 'Location[T.UK]*Photoperiod'])

plt.show()

In [ ]:
# QQ-plot to verify the residuals of the model
plt.figure(figsize=(8, 6))
sm.qqplot(model5b.resid, line='45')
plt.title('QQ-plot of the residuals')
plt.show()

In [ ]:
residualsX = model5b.resid 
predictedX = model5b.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_testX = het_breuschpagan(residualsX, model5b.model.exog)

# results of the Breusch-Pagan test
bp_statX, bp_pvalX, _, _ = bp_testX
print(f'Breusch-Pagan statistic: {bp_statX}, p-value: {bp_pvalX}')
if bp_pvalX > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_testX = durbin_watson(residualsX)

print('Durbin-Watson test:', durbin_watson_testX)

In [ ]:
# Order the dat to fit the GEE model
df2 = df2.sort_values(by='date')

# define the structure of the GEE model(Autoregressive) to capture the temporal dependence
cov_str = sm.cov_struct.Autoregressive() # For longitudinal data: proximate data points exhibit stronger correlation than distant data points

# GEE model with random intercepts for each date
gee_model = smf.gee(
    'phase ~ C(location) + C(DST_1) + day_after_flight + photoperiod + C(location)*photoperiod + C(location)*day_after_flight',
    data=df2,
    groups=df2['flight_id'],  # to model the intra-day post-flight correlation: 
                                     # if the day after flight affects sleep in a systematic way and if observations from the same day_after_flight are more similar to each other than observations from different days
    cov_struct=cov_str,
    family=sm.families.Gaussian()
).fit()

gee_model.summary()

In [ ]:
plt.figure(figsize=(18, 6))
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df2, palette=['darkblue', 'darkorange'])
plt.title('Phase by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (wake up time-sunrise)')
plt.xlabel('')
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5)) 

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

In [ ]:
# Create a figure with two subplots (one above the other) with adjusted height ratios
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 9), sharex=True, gridspec_kw={'height_ratios': [5, 5]})

# Plot the phase by location over time in the top subplot
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df2, palette=['darkblue', 'darkorange'], ax=ax1)
ax1.set_title('Phase (wake up time-sunrise) by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
ax1.set_ylabel('Phase (h)')
ax1.set_xlabel('')
ax1.legend()
ax1.grid(True)
ax1.tick_params(axis='x', rotation=55)
ax1.set_xlim(df2['date'].min(), df2['date'].max())
ax1.xaxis.set_major_locator(MultipleLocator(28))  # Set the x-axis major locator

#  set the legend outside the plot
ax1.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))

# Add vertical bars to indicate the start of DST and ST
dst_st_dates = [
    ('2022-10-30', 'start ST'),
    ('2023-10-29', 'start ST'),
    ('2023-03-26', 'start DST'),
    ('2024-03-31', 'start DST'),
    ('2024-10-27', 'start ST')
]
for date, label in dst_st_dates:
    ax1.axvline(x=pd.to_datetime(date), color='black', linestyle='--')
    ax1.text(pd.to_datetime(date), ax1.get_ylim()[0], label, ha='right', va='bottom')

# Define seasons and apply background color for each season
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
    ('2024-12-22', '2025-03-20', '#335f9e'),  # Winter
]
for start, end, color in seasons:
    ax1.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

# Annotate the seasons
season_annotations = [
    ('Summer', 1.0555, 0.79, 'gold'),
    ('Autumn', 1.0535, 0.97, 'chocolate'),
    ('Winter', 1.05, 0.91, '#76d4f5'),
    ('Spring', 1.0499, 0.85, '#95de74')
]
for label, x, y, color in season_annotations:
    ax1.annotate(label, xy=(x, y), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor=color)

# Plot the sunrise and sunset times in the bottom subplot
sns.lineplot(x='date', y='sunrise (uk), hours_adjust', data=df_daylight1, ax=ax2, label='Sunrise (UK)', color='orange')
sns.lineplot(x='date', y='sunset (uk), hours_adjust', data=df_daylight1, ax=ax2, label='Sunset (UK)', color='purple')
sns.lineplot(x='date', y='sunrise (ita), hours_adjust', data=df_daylight1, ax=ax2, label='Sunrise (Italy)', color='red')
sns.lineplot(x='date', y='sunset (ita), hours_adjust', data=df_daylight1, ax=ax2, label='Sunset (Italy)', color='blue')

ax2.set_ylabel('Time (h, local time)')
ax2.set_xlabel('')
# Format y-axis to hh:mm
ax2.yaxis.set_major_formatter(FuncFormatter(hours_to_hhmm))
ax2.grid(True)
ax2.tick_params(axis='x', rotation=55)
ax2.set_xlim(df2['date'].min(), df2['date'].max())
ax2.xaxis.set_major_locator(MultipleLocator(28))  # Set the x-axis major locator

#  set the legend outside the plot
ax2.legend(loc='upper left', bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()


Wake After Sleep Onset

In [ ]:
# drop the nan values in the waso_min column
df_waso_clean = df.dropna(subset=['waso'])
df_waso_clean_x = df_flight1.dropna(subset=['waso'])

In [ ]:
model1a = smf.mixedlm('waso ~ day_after_flight', df_waso_clean, groups=df_waso_clean['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model1a.summary())

In [ ]:
model2 = smf.mixedlm('waso ~ day_after_flight + C(location) + C(location)*day_after_flight', df_waso_clean, groups=df_waso_clean['flight_id']).fit(method='powell', maxiter=1000) # different optimization methods to provide better convergence: common methods include 'lbfgs', 'cg', 'powell', and 'bfgs'.

print(model2.summary())

In [ ]:
#interpretation of the model
# location[T.UK] = 0.0: the reference location is Italy
# day_after_flight = 0.0: the reference day after flight is 1-2 days after the flight
# location[T.UK]: the effect of the UK location on the waso is 0.0 minutes
# day_after_flight: the effect of the day after flight on the waso is 0.0 minutes   
# location[T.UK]:day_after_flight: the interaction effect of the UK location and the day after flight on the waso is 0.0 minutes this means that the effect of the UK location on the waso is the same for all days after the flight
# the p-value of the location[T.UK]:day_after_flight is 0.0 which is less than 0.05, this means that the interaction effect of the UK location and the day after flight on the waso is significant and this means that in the UK the waso is different for each day after the flight

In [ ]:
shapiro_test_midsleep = stats.shapiro(df_waso_clean_x['waso'])

shapiro_test_midsleep

In [ ]:
utest_waso_days = stats.mannwhitneyu(df_waso_clean_x[df_waso_clean_x['day_after_flight_group'] == '0']['waso'], df_waso_clean_x[df_waso_clean_x['day_after_flight_group'] == '1']['waso'])

utest_waso_days

In [ ]:
# mean and std of midsleep by day_after_flight_group
df_grouped_flight = df_waso_clean_x.groupby(['day_after_flight_group']).agg({'waso': ['mean', 'std']})
df_grouped_flight

In [ ]:
# Fit the factorial ANOVA model
model4 = ols('waso ~ C(location) * C(day_after_flight_group)', data=df_waso_clean_x).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model4, typ=2)

print(anova_results)

_Analysis of the subset: January 2023-2024-2025_

In [ ]:
df_2025 = pd.read_excel(fpath + '\\10.0_database_variables.xlsx', sheet_name='v2database2022_2025')

In [ ]:
# extract january data
df_january23 = df[(df['date'] >= '2023-01-01') & (df['date'] <= '2023-01-31')]
df_january24 = df[(df['date'] >= '2024-01-01') & (df['date'] <= '2024-01-31')]
df_january25 = df_2025[(df_2025['date'] >= '2025-01-01') & (df_2025['date'] <= '2025-01-31')]

In [ ]:
df_january23 = df_january23.drop(['sleep_duration_free_days', 'sleep_duration_work_days', 'sleep_end_decimal_plot', 'sleep_end_decimal_UTC_plot', 'photoperiod_tertile', 'week_of_year', 'season'], axis=1)  
df_january24 = df_january24.drop(['sleep_duration_free_days', 'sleep_duration_work_days', 'sleep_end_decimal_plot', 'sleep_end_decimal_UTC_plot', 'photoperiod_tertile', 'week_of_year', 'season'], axis=1)

In [ ]:
# rename the location column as 0=ITA, 1=UK
df_january25['location'] = df_january25['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df_january25['weekday_type'] = df_january25['weekday_type'].map({0: 'work days', 1: 'free days'})

In [ ]:
# new variable 'photoperiod' based on the location
# if column 'location' = 1 take the value from 'photoperiod (h, UK)' 
# if column 'location' = 0 then photoperiod (h, ITA)'
df_january25['photoperiod'] = np.where(df_january25['location'] == 'UK', df_january25['photoperiod (h, UK)'], df_january25['photoperiod (h, ITA)'])

In [ ]:
# concatenate the two dataframes
df_january = pd.concat([df_january23, df_january24, df_january25], ignore_index=True)

In [ ]:
# add a column if date 2025=1 and 2024=0
df_january['year'] = df_january['date'].dt.year

In [ ]:
# if location=1 , delete row
df_january_ita = df_january[df_january['location'] != 'UK']

In [ ]:
df_january.groupby('year').describe()

#print as a table
df_january.groupby('year').describe().T

In [ ]:
# count the day for jan 2023, 2024, 2025
df_january['year'].value_counts()

In [ ]:
anova_jan_midsleep = ols('midsleep_h ~ C(year)', data=df_january).fit() # generate and fit the regression model
anovaresults_jan_midsleep = sm.stats.anova_lm(anova_jan_midsleep, typ=3)

In [ ]:
print('ANOVA Result for midsleep:')
print(anovaresults_jan_midsleep)

In [ ]:
# mean and std of midsleep by year
midsleep_jan_year = df_january.groupby(['year']).agg({'midsleep_h': ['mean', 'std']})

print(midsleep_jan_year)

In [ ]:
posthoc_midsleep = pairwise_tukeyhsd(df_january['midsleep_h'], df_january['year'])
print(posthoc_midsleep)

In [ ]:
anova_jan_sleepduration = ols('sleep_duration ~ C(year)', data=df_january).fit() # generate and fit the regression model
anovaresults_jan_sleepduration = sm.stats.anova_lm(anova_jan_sleepduration, typ=3)

In [ ]:
print('ANOVA Result for midsleep:')
print(anovaresults_jan_sleepduration)

In [ ]:
# sleep onset kuskal wallis test
kw_jan_sleep_onset = stats.kruskal(df_january[df_january['year'] == 2023]['sleep_start_decimal'], df_january[df_january['year'] == 2024]['sleep_start_decimal'], df_january[df_january['year'] == 2025]['sleep_start_decimal'])
# sleep end kuskal wallis test
kw_jan_sleep_end = stats.kruskal(df_january[df_january['year'] == 2023]['sleep_end_decimal'], df_january[df_january['year'] == 2024]['sleep_end_decimal'], df_january[df_january['year'] == 2025]['sleep_end_decimal'])
# waso kruskal wallis test
kw_jan_waso = stats.kruskal(df_january[df_january['year'] == 2023]['waso'], df_january[df_january['year'] == 2024]['waso'], df_january[df_january['year'] == 2025]['waso'])

In [ ]:
# print the results
print(f"Sleep onset: {kw_jan_sleep_onset}")
print(f"Sleep offset: {kw_jan_sleep_end}")
print(f"WASO: {kw_jan_waso}")

In [ ]:
posthoc_sleepoffset = pairwise_tukeyhsd(df_january['sleep_end_decimal'], df_january['year'])

print(posthoc_sleepoffset)

In [ ]:
# mean and std of slep offset by year
sleepoffset_jan_year = df_january.groupby(['year']).agg({'sleep_end_decimal': ['mean', 'std']})

print(sleepoffset_jan_year)

In [ ]:
anova_jan_midsleep_ita = ols('midsleep_h ~ C(year)', data=df_january_ita).fit() # generate and fit the regression model
anovaresults_jan_midsleep_ita = sm.stats.anova_lm(anova_jan_midsleep_ita, typ=3)

In [ ]:
print('ANOVA Result for midsleep:')
print(anovaresults_jan_midsleep_ita)

In [ ]:
anova_jan_sleepduration_ita = ols('sleep_duration ~ C(year)', data=df_january_ita).fit() # generate and fit the regression model
anovaresults_jan_sleepduration_ita = sm.stats.anova_lm(anova_jan_sleepduration_ita, typ=3)

In [ ]:
print('ANOVA Result for midsleep:')
print(anovaresults_jan_sleepduration_ita)

In [ ]:
# sleep onset kuskal wallis test
kw_jan_sleep_onset_ita = stats.kruskal(df_january_ita[df_january_ita['year'] == 2023]['sleep_start_decimal'], df_january_ita[df_january_ita['year'] == 2024]['sleep_start_decimal'], df_january_ita[df_january_ita['year'] == 2025]['sleep_start_decimal'])
# sleep end kuskal wallis test
kw_jan_sleep_end_ita = stats.kruskal(df_january_ita[df_january_ita['year'] == 2023]['sleep_end_decimal'], df_january_ita[df_january_ita['year'] == 2024]['sleep_end_decimal'], df_january_ita[df_january_ita['year'] == 2025]['sleep_end_decimal'])
# waso kruskal wallis test
kw_jan_waso_ita = stats.kruskal(df_january_ita[df_january_ita['year'] == 2023]['waso'], df_january_ita[df_january_ita['year'] == 2024]['waso'], df_january_ita[df_january_ita['year'] == 2025]['waso'])

In [ ]:
# print the results
print(f"Sleep onset: {kw_jan_sleep_onset_ita}")
print(f"Sleep offset: {kw_jan_sleep_end_ita}")
print(f"WASO: {kw_jan_waso_ita}")

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='midsleep_h', data=df_january)
plt.title('Midsleep by year ', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='sleep_end_decimal', data=df_january)
plt.title('Sleep offset by year ', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep offset (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# one way anova with the photoperiod between 2023, 2024, 2025
anova_jan_photoperiod = stats.f_oneway(df_january_ita[df_january_ita['year'] == 2023]['photoperiod'], df_january_ita[df_january_ita['year'] == 2024]['photoperiod'], df_january_ita[df_january_ita['year'] == 2025]['photoperiod'])

print(anova_jan_photoperiod)

In [ ]:
#plot the photoperiod across the years
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='photoperiod', data=df_january, hue='location')
plt.title('Photoperiod by year', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Photoperiod (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.ylim(7.8, 9.9)
plt.tight_layout()
plt.show()

In [ ]:
# Fit the factorial ANOVA model
model_jan1 = ols('sleep_end_decimal ~ year * weekday_type', data=df_january_ita).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model_jan1, typ=2)

print(anova_results)

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='year', y='sleep_end_decimal', data=df_january, hue='weekday_type')
plt.title('Sleep offset by year and weekday type', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Sleep offset (h)')
plt.xlabel('')
plt.grid(False)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 0.5))
plt.show()

In [ ]:
# Fit the factorial ANOVA model
model_jan2 = ols('midsleep_h ~ year * weekday_type', data=df_january).fit()

# Perform ANOVA
anova_results = sm.stats.anova_lm(model_jan2, typ=2)

print(anova_results)